In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/spanish-housing-dataset-location-size-price-and")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\pibanezmor\.cache\kagglehub\datasets\thedevastator\spanish-housing-dataset-location-size-price-and\versions\2


In [11]:
import kagglehub
import pandas as pd
import os

# 1. Descargar el dataset completo
print("Descargando dataset...")
path = kagglehub.dataset_download("thedevastator/spanish-housing-dataset-location-size-price-and")
print("Ruta de los archivos:", path)

# 2. Apuntar directamente al archivo de España
ruta_casas = os.path.join(path, 'spanish_houses.csv')

# 3. Cargar los datos en bruto
print("Cargando el archivo original de toda España...")
df_bruto = pd.read_csv(ruta_casas)

# 4. Exportar el CSV tal cual, sin tocar nada
nombre_archivo = 'SPANISH_HOUSES_BRUTO.csv'
df_bruto.to_csv(nombre_archivo, index=False)

print(f"--- ¡LISTO! ---")
print(f"Tu archivo se ha guardado como: {nombre_archivo}")
print(f"Total de viviendas (filas): {len(df_bruto)}")
print(f"Total de características (columnas): {len(df_bruto.columns)}")

# Echamos un vistazo a las 5 primeras filas y a todas sus columnas
display(df_bruto.head())

Descargando dataset...
Ruta de los archivos: C:\Users\pibanezmor\.cache\kagglehub\datasets\thedevastator\spanish-housing-dataset-location-size-price-and\versions\2
Cargando el archivo original de toda España...
--- ¡LISTO! ---
Tu archivo se ha guardado como: SPANISH_HOUSES_BRUTO.csv
Total de viviendas (filas): 100000
Total de características (columnas): 41


,ad_description,ad_last_update,air_conditioner,balcony,bath_num,built_in_wardrobe,chimney,condition,construct_date,energetic_certif,...,room_num,storage_room,swimming_pool,terrace,unfurnished,number_of_companies_prov,population_prov,companies_prov_vs_national_%,population_prov_vs_national_%,renta_media_prov
0,Precio chalet individual en la localidad de Ab...,Anuncio actualizado el 27 de marzo,0,0,2,0,0,segunda mano/buen estado,NaN,NaN,...,4,0,0,1,NaN,19147,328868,0.57,0.7,19889.0
1,"Atico de 80m2, para entrar a vivir, con salón ...",más de 5 meses sin actualizar,0,0,2,0,0,segunda mano/buen estado,2006.0,no indicado,...,3,1,0,0,NaN,19147,328868,0.57,0.7,19889.0
2,B/ Etxaguen. Casa de reciente construcción con...,más de 5 meses sin actualizar,0,0,3,0,0,segunda mano/buen estado,NaN,no indicado,...,4,1,0,1,NaN,19147,328868,0.57,0.7,19889.0
3,Se vende vivienda en abornikano (ayuntamiento ...,más de 5 meses sin actualizar,0,1,1,1,1,segunda mano/buen estado,NaN,en trámite,...,4,1,0,1,NaN,19147,328868,0.57,0.7,19889.0
4,Negociables.,más de 5 meses sin actualizar,0,0,1,0,0,segunda mano/buen estado,NaN,no indicado,...,2,1,1,1,NaN,19147,328868,0.57,0.7,19889.0


In [12]:
import kagglehub
import pandas as pd
import os

# 1. Descargamos los datos desde Kaggle
path = kagglehub.dataset_download("thedevastator/spanish-housing-dataset-location-size-price-and")
ruta_casas = os.path.join(path, 'spanish_houses.csv')

# 2. Leemos el archivo bruto de todas las casas de España
df_bruto = pd.read_csv(ruta_casas)

# 3. SACAMOS LA COPIA EN CSV
nombre_del_archivo = 'COPIA_SPANISH_HOUSES.csv'
df_bruto.to_csv(nombre_del_archivo, index=False, encoding='utf-8')

print(f"¡Listo! Se ha creado una copia exacta de los datos.")
print(f"Busca el archivo '{nombre_del_archivo}' en la misma carpeta donde tienes este código.")

¡Listo! Se ha creado una copia exacta de los datos.
Busca el archivo 'COPIA_SPANISH_HOUSES.csv' en la misma carpeta donde tienes este código.


In [15]:
import pandas as pd
import re

# 1. Cargar el Mega-CSV en bruto
print("Cargando datos...")
df_bruto = pd.read_csv('COPIA_SPANISH_HOUSES.csv', low_memory=False)


Cargando datos...


In [16]:

# 2. Seleccionar las columnas estrella
columnas_estrella = [
    'loc_zone', 'renta_media_prov', # Localización
    'm2_real', 'room_num', 'bath_num', 'condition', 'floor', # Físicas
    'lift', 'garage', 'swimming_pool', 'air_conditioner', 'terrace', # Extras
    'price' # Objetivo
]
df_red = df_bruto[columnas_estrella].copy()


In [17]:

# 3. LIMPIEZA DE NÚMEROS Y NULOS BÁSICA
for col in ['price', 'm2_real', 'room_num', 'bath_num', 'renta_media_prov']:
    df_red[col] = pd.to_numeric(df_red[col], errors='coerce')

df_red = df_red.dropna(subset=['price', 'm2_real', 'loc_zone'])


In [ ]:

# 4. TRATAMIENTO DE LOCALIZACIÓN
# Calculamos el precio medio de cada zona y lo guardamos en un diccionario
precio_medio_por_zona = df_red.groupby('loc_zone')['price'].mean().to_dict()

# Reemplazamos el nombre de la zona (texto) por su precio medio (número)
# Lo llamamos 'loc_zone_value'
df_red['loc_zone_value'] = df_red['loc_zone'].map(precio_medio_por_zona)

# Ya podemos borrar la columna de texto original
df_red = df_red.drop(columns=['loc_zone'])


In [22]:

# 5. CONVERSIÓN DE TEXTO A NÚMERO (Floor y Condition)
# Extraer número de planta
def limpiar_planta(x):
    if pd.isna(x): return 0
    nums = re.findall(r'\d+', str(x))
    return int(nums[0]) if nums else 0
df_red['floor'] = df_red['floor'].apply(limpiar_planta)

# Mapear estado
mapping_cond = {'segunda mano/para reformar': 0, 'segunda mano/buen estado': 1, 'promoción de obra nueva': 2}
df_red['condition'] = df_red['condition'].map(mapping_cond).fillna(1).astype(int)

# --- NUEVA VERSIÓN PARA LOS EXTRAS ---
extras = ['lift', 'garage', 'swimming_pool', 'air_conditioner', 'terrace']

def limpiar_extras_letras(x):
    # 1. Si está totalmente vacío (NaN), ponemos un 0
    if pd.isna(x): 
        return 0
    
    # 2. Convertimos a texto para analizarlo
    texto = str(x).strip()
    
    # 3. Si el dato era explícitamente un cero ("0" o "0.0"), lo mantenemos como 0
    if texto in ['0', '0.0']:
        return 0
        
    # 4. Si hay CUALQUIER otra cosa (letras, frases, "1", "1.0"), ponemos un 1
    return 1

# Aplicamos la función a todas las columnas de extras
for col in extras:
    df_red[col] = df_red[col].apply(limpiar_extras_letras)


In [24]:

# 6. ELIMINAR CUALQUIER NULO RESTANTE Y EXPORTAR
df_red = df_red.dropna()
df_red.to_csv('DATASET_RED_NEURONAL_LISTO.csv', index=False)

print("--- ¡DATASET LISTO PARA LA IA! ---")
print(f"Filas finales: {len(df_red)}")
print("Todas las columnas son ahora NUMÉRICAS:")
print(df_red.dtypes)
display(df_red.head())

--- ¡DATASET LISTO PARA LA IA! ---
Filas finales: 13846
Todas las columnas son ahora NUMÉRICAS:
renta_media_prov    float64
m2_real             float64
room_num            float64
bath_num            float64
condition             int64
floor                 int64
lift                  int64
garage                int64
swimming_pool         int64
air_conditioner       int64
terrace               int64
price               float64
loc_zone_value      float64
dtype: object


,renta_media_prov,m2_real,room_num,bath_num,condition,floor,lift,garage,swimming_pool,air_conditioner,terrace,price,loc_zone_value
4,19889.0,76.0,2.0,1.0,1,1,1,1,1,0,1,90000.0,326596.686567
6,19889.0,100.0,3.0,2.0,1,1,1,1,0,0,1,195000.0,326596.686567
13,19889.0,70.0,2.0,2.0,1,1,0,1,0,0,0,150000.0,326596.686567
23,19889.0,108.0,3.0,2.0,1,1,0,1,0,0,0,160000.0,245094.414324
47,19889.0,78.0,2.0,2.0,1,1,1,1,1,0,1,175000.0,326596.686567
